# Variabilidade, intervalos e testes de hipótese

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/02_incerteza_e_testes_de_hipotese_revisado.ipynb)

Nos notebooks anteriores, aprendemos a **comparar grupos** e calcular diferenças observadas. Agora enfrentaremos uma pergunta que aparece frequentemente no noticiário:

> **Quando uma pesquisa eleitoral indica 39% de intenção de voto, com margem de erro de dois pontos percentuais e nível de confiança de 95%, o que esses números significam?**

Uma pesquisa é realizada com uma parte da população. Seu resultado pode mudar quando outras pessoas são entrevistadas, mas nem toda diferença decorre da escolha da amostra. O modo de selecionar participantes, formular perguntas e registrar respostas também importa.

A pergunta norteadora passa a ser:

> **Que variação esperaríamos para nossa estimativa se o procedimento de obtenção dos dados pudesse ser repetido?**

Faremos dois movimentos: primeiro, interpretar **pesquisas reais divulgadas antes do primeiro turno presidencial de 2026**; depois, utilizar **simulações explicitamente didáticas** e o corpus sintético de 120 documentos da Unidade 5 para compreender variabilidade, *bootstrap*, intervalos e testes.

## 1. Uma pesquisa eleitoral: três números que não dizem a mesma coisa

Na pesquisa Quaest divulgada em 28 de setembro de 2026, Lula registrou **39%** das intenções de voto no cenário estimulado de primeiro turno. A margem de erro divulgada foi de **2 pontos percentuais**, para mais ou para menos, com **95% de confiança**.

A leitura introdutória é: **39%** é uma estimativa pontual; **±2 pontos percentuais** é a margem divulgada; **95%** é o nível de confiança declarado. Uma visualização simplificada dos limites é **37% a 41%**, *sob os pressupostos metodológicos do instituto*.

Isso **não** significa que 95% dos eleitores concordem com a pesquisa, nem que exista 95% de probabilidade frequentista de o percentual populacional fixo estar no intervalo já calculado. A interpretação correta será construída ao longo da aula.

**Antes de executar qualquer célula**, registre uma hipótese: se fizéssemos outra pesquisa igualmente planejada, o resultado seria exatamente 39%? Por quê?

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import os
import subprocess

URL_REPOSITORIO = "https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais.git"
REPOSITORIO = Path("/dados/disciplina_computacao_aplicada_humanidades_digitais")
PASTA_UNIDADE = REPOSITORIO / "unidade_05"

try:
    import google.colab  # type: ignore
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        REPOSITORIO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             URL_REPOSITORIO, str(REPOSITORIO)],
            check=True,
        )
    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

### Três levantamentos publicados: Quaest, Datafolha e AtlasIntel

A comparação será restrita ao **primeiro turno, intenção de voto estimulada, percentuais sobre o total de entrevistados**, com pesquisas divulgadas em setembro de 2026. Não misturaremos percentuais de votos válidos, de segundo turno ou de perguntas espontâneas.

Os dados abaixo foram transcritos das divulgações públicas; **não são respostas individuais dos entrevistados**. Os registros do TSE são identificadores de pesquisa, e a publicação de resultados e suas condições metodológicas deve ser conferida nos documentos de cada registro.

**Fontes consultadas:**

- [Quaest — resultados publicados pela Folha](https://www1.folha.uol.com.br/poder/2026/09/quaest-lula-tem-39-e-flavio-bolsonaro-34-no-1o-turno.shtml); [informações de campo divulgadas pela Agência Estado/UOL](https://noticias.uol.com.br/ultimas-noticias/agencia-estado/2026/09/28/aprovacao-de-lula-oscila-de-44-a-46-aponta-quaest-desaprovacao-vai-de-50-a-49.amp.htm).
- [Datafolha — página do instituto e relatório da pesquisa](https://datafolha.folha.uol.com.br/eleicoes/2026/09/no-1o-turno-lula-pt-tem-40-e-flavio-bolsonaro-pl-aparece-com-36.shtml).
- [AtlasIntel — página do instituto](https://atlasintel.org/poll/brazil-national-2026-09-23); [percentuais publicados pela CNN Brasil](https://www.cnnbrasil.com.br/eleicoes/atlas-bloomberg-lula-tem-458-no-1o-turno-flavio-434/).
- [Portal de Dados Abertos do TSE — pesquisas eleitorais 2026](https://dadosabertos.tse.jus.br/dataset/pesquisas-eleitorais-2026).

**Nota de auditoria:** as fontes secundárias consultadas não são uniformes sobre o último dia de campo da Quaest: uma reportagem da Folha indica 28/09, enquanto Agência Estado/UOL e outras fichas indicam **27/09**. Usaremos 24–27/09 na tabela, deixando essa divergência documentada até conferir a ficha primária. A divergência **não altera** os percentuais aqui transcritos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Transcrição congelada de pesquisas públicas: NÃO são microdados.
dados_pesquisas = pd.DataFrame([
    {"instituto": "Quaest", "coleta": "24–27/set*", "n": 2004,
     "modo": "presencial", "Lula": 39.0, "Flávio": 34.0,
     "margem_pp": 2.0, "confianca": 95, "registro": "BR-06520/2026"},
    {"instituto": "Datafolha", "coleta": "22–23/set", "n": 2002,
     "modo": "presencial", "Lula": 40.0, "Flávio": 36.0,
     "margem_pp": 2.0, "confianca": 95, "registro": "BR-00304/2026"},
    {"instituto": "AtlasIntel", "coleta": "17–22/set", "n": 5015,
     "modo": "digital", "Lula": 45.8, "Flávio": 43.4,
     "margem_pp": 1.0, "confianca": 95, "registro": "BR-04739/2026"},
])

dados_pesquisas["dif_Lula_menos_Flavio_pp"] = (
    dados_pesquisas["Lula"] - dados_pesquisas["Flávio"]
)
display(dados_pesquisas)
print("* O último dia de campo da Quaest diverge entre fontes; verificar ficha no TSE.")

### Um intervalo divulgado não é uma garantia

Para praticar a leitura, somaremos e subtrairemos a margem publicada de **cada estimativa individual**. Essas faixas servem à leitura do anúncio: **não são intervalos reconstruídos a partir dos microdados**, nem incorporam todas as fontes de erro.

**Questão rápida:** Quaest e Datafolha anunciaram margens de 2 p.p. e obtiveram estimativas diferentes. Qual das afirmações é defensável?

- Os institutos obrigatoriamente deveriam publicar a mesma estimativa.
- A margem publicada captura necessariamente todos os efeitos de métodos, perguntas e datas diferentes.
- Os resultados podem diferir por variabilidade amostral e por outras decisões ou condições de pesquisa.

**A terceira é a resposta defensável.** Também não se deve testar a diferença **entre dois candidatos na mesma pesquisa** simplesmente somando, subtraindo ou comparando os limites marginais: estimativas provenientes dos mesmos entrevistados são dependentes. A diferença precisa de incerteza calculada para a **estatística da diferença** e do desenho amostral adequado.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
posicoes = np.arange(len(dados_pesquisas))
for deslocamento, candidato in [(-0.14, "Lula"), (0.14, "Flávio")]:
    ax.errorbar(
        dados_pesquisas[candidato], posicoes + deslocamento,
        xerr=dados_pesquisas["margem_pp"],
        fmt="o", capsize=4, label=candidato,
    )
ax.set_yticks(posicoes, dados_pesquisas["instituto"])
ax.invert_yaxis()
ax.set_xlabel("Intenção de voto estimulada (%)")
ax.set_title("Estimativas divulgadas e margens anunciadas — setembro de 2026")
ax.legend()
ax.grid(axis="x", alpha=0.2)
plt.show()
print("Atenção: as barras são as margens publicadas, NÃO ICs reconstruídos e NÃO testes da diferença entre candidatos.")

## 2. Por que uma amostra pode ser problemática?

Antes de calcular um intervalo, precisamos perguntar **como os dados foram produzidos**. Os termos abaixo designam problemas diferentes:

| Fonte de problema ou variação | Exemplo em pesquisa eleitoral | O que uma amostra maior resolve? |
|---|---|---|
| **Erro de medição** | Uma formulação ambígua induz respostas diferentes da opinião que se pretendia medir | Não necessariamente; pode reproduzir a mesma distorção |
| **Transcrição ou codificação** | Uma resposta digitada na categoria errada | Não, se o procedimento continuar incorreto |
| **Lacuna de cobertura** | Parte do eleitorado tem dificuldade de ser alcançada pelo meio de coleta | Não, se continuar excluída |
| **Viés de seleção ou não resposta** | A propensão a participar está associada à preferência eleitoral | Não, se os mecanismos permanecerem |
| **Mudança temporal real** | Eleitores mudam de opinião entre o campo e o dia da votação | Não é erro da estimativa relativa à data do campo |
| **Operacionalização** | A pergunta espontânea e a estimulada medem respostas a estímulos diferentes | Não; é necessário definir a pergunta correta |

Uma margem calculada apenas a partir da **variabilidade amostral** não quantifica automaticamente todos esses problemas. A precisão numérica não substitui a validade da coleta.

### Diagnóstico, antes dos cálculos

Imagine uma pesquisa feita apenas com pessoas que respondem a um formulário divulgado em um perfil de rede social. Ela alcança 50 mil respondentes e anuncia uma margem extremamente pequena. **Esse número de respondentes, sozinho, torna a pesquisa representativa?** Registre uma justificativa antes de avançar.

## 3. Experimento: uma população conhecida e muitas pesquisas

Nas pesquisas reais, não sabemos o percentual exato de intenção de voto da população. Na simulação, vamos **definir artificialmente** que 42% de uma população hipotética apoiam uma opção chamada *A*. Esse número é inventado e **não representa Lula, Flávio ou qualquer outro candidato real**.

Agora podemos observar centenas de pesquisas independentes sob duas situações:

- **Seleção idealizada:** cada observação é uma amostra independente da mesma população; a probabilidade de resposta de interesse é 42%.
- **Não resposta seletiva fictícia:** pessoas com preferência A participam menos; a amostra disponível fica sistematicamente distorcida.

Para a primeira situação, ilustraremos uma fórmula aproximada de margem para uma proporção sob amostragem aleatória simples independente:

$$
\widehat{p}\;\pm\;1{,}96\sqrt{\frac{\widehat{p}(1-\widehat{p})}{n}}.
$$

**Importante:** esta fórmula é uma aproximação didática para a simulação. Ela **não** reproduz as margens anunciadas pelos três institutos, que podem envolver ponderação, estratificação, desenho amostral e convenções metodológicas próprias.

In [ ]:
SEMENTE_SIMULACAO = 20261015
rng_sim = np.random.default_rng(SEMENTE_SIMULACAO)
p_populacao = 0.42  # Cenário inventado: não é um percentual de candidato real.
repeticoes = 500

def repeticoes_de_pesquisa(n, p_respostas, repeticoes=500):
    estimativas = rng_sim.binomial(n, p_respostas, size=repeticoes) / n
    margem = 1.96 * np.sqrt(estimativas * (1 - estimativas) / n)
    return estimativas, margem

linhas_experimento = []
for n in [400, 2000, 5000]:
    estimativas, margens = repeticoes_de_pesquisa(n, p_populacao, repeticoes)
    cobertos = ((estimativas - margens <= p_populacao)
                & (p_populacao <= estimativas + margens))
    linhas_experimento.append({
        "n": n,
        "média das estimativas (%)": estimativas.mean() * 100,
        "margem média (p.p.)": margens.mean() * 100,
        "cobertura empírica (%)": cobertos.mean() * 100,
    })

resumo_simulacao = pd.DataFrame(linhas_experimento)
display(resumo_simulacao.round(2))

In [ ]:
n_exemplo = 400
estimativas, margens = repeticoes_de_pesquisa(n_exemplo, p_populacao, 65)
y = np.arange(len(estimativas))
fig, ax = plt.subplots(figsize=(10, 7))
ax.errorbar(estimativas * 100, y, xerr=margens * 100,
            fmt="o", capsize=2, markersize=3)
ax.axvline(p_populacao * 100, linestyle="--", label="valor verdadeiro (simulado)")
ax.set_yticks([])
ax.set_xlabel("Estimativa e intervalo aproximado (%)")
ax.set_ylabel("65 pesquisas fictícias independentes")
ax.set_title("Mesmo procedimento, respostas diferentes, intervalos diferentes")
ax.legend()
plt.show()

### E se a seleção for sistematicamente distorcida?

Agora suporemos, **somente na simulação**, que apoiadores de A têm probabilidade de participar proporcional a 0,55, enquanto os demais têm propensão 0,95. A composição dos respondentes deixa de refletir diretamente a população inventada.

Para obter a proporção esperada de A **entre os respondentes**, usamos:

$$
p_{\mathrm{respondentes}}=
\frac{0{,}42\times0{,}55}{0{,}42\times0{,}55+0{,}58\times0{,}95}.
$$

Esse modelo é extremamente simplificado e não descreve o processo de nenhum dos institutos reais. Sua utilidade é mostrar que **aumentar o tamanho de uma amostra enviesada pode reduzir a margem calculada sem eliminar a distorção**.

In [ ]:
propensao_a, propensao_outros = 0.55, 0.95
p_entre_respondentes = (
    p_populacao * propensao_a
    / (p_populacao * propensao_a + (1-p_populacao) * propensao_outros)
)
resultado_vies = []
for n in [400, 2000, 5000]:
    estimativas, margens = repeticoes_de_pesquisa(n, p_entre_respondentes, 500)
    resultado_vies.append({
        "n": n,
        "estimativa média (%)": estimativas.mean() * 100,
        "margem média (p.p.)": margens.mean() * 100,
        "erro em relação à população (p.p.)": (estimativas.mean() - p_populacao) * 100,
    })
print("Proporção verdadeira na população fictícia: 42%")
print(f"Proporção esperada entre os respondentes fictícios: {p_entre_respondentes*100:.1f}%")
display(pd.DataFrame(resultado_vies).round(2))

### Experimento especial: repetição de pesquisas e cobertura de 95%

A população sintética de **1 milhão de eleitores** (v0.7/v0.8/v1.1) possui preferências fictícias conhecidas: **A = 41,9945%**, **B = 38,0190%**, **C = 12,0219%** e outras respostas. Podemos repetir milhares de pesquisas sobre **a mesma população fixa**.

Cada pesquisa produz, para cada candidato, uma **estimativa pontual** e um **intervalo de confiança**. Contamos quantos intervalos contêm a proporção verdadeira da população. Após \(R\) pesquisas, a **cobertura acumulada** é

\[
\widehat{C}_R=\frac{1}{R}\sum_{r=1}^{R}\mathbf{1}\{L_r\leq p_{\mathrm{verdadeiro}}\leq U_r\}.
\]

Se o intervalo for calibrado para o desenho e as hipóteses forem satisfeitas, essa frequência tende à **cobertura real do procedimento**, idealmente próxima de 95%. **Não se espera uma aproximação monotônica nem uma cobertura exatamente igual a 95%.** Em um método enviesado, a frequência pode convergir para um valor muito inferior.

Os nomes *domiciliar*, *pontos de fluxo* e *digital* representam **modelos didáticos**, não reconstituições dos procedimentos de Quaest, Datafolha ou AtlasIntel. Nos métodos por cotas e digitais, os intervalos são *pseudo-intervalos* cuja cobertura apenas observamos, sem garantia nominal.

In [ ]:
# Execução autossuficiente: controle AAS e hipótese simplificada de não resposta.
# Para comparar também os cinco métodos ancorados na população de 1 milhão,
# execute a célula seguinte com o CSV de rodadas do pacote v1.5.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CANDIDATOS = ["A", "B", "C", "OUTRA", "INDECISO", "BRANCO_NULO"]
VERDADE = np.array([0.419945, 0.380190, 0.120219, 0.029968, 0.029766, 0.019912])
assert abs(VERDADE.sum()-1) < 1e-12
NUM_REPETICOES = 3000
ENTREVISTAS_POR_PESQUISA = 2000
rng_cobertura = np.random.default_rng(202610161015)

def intervalo_wilson(contagens, n, z=1.959963984540054):
    prop = contagens / n
    den = 1 + z*z/n
    centro = (prop + z*z/(2*n)) / den
    semi = z / den * np.sqrt(prop*(1-prop)/n + z*z/(4*n*n))
    return centro-semi, centro+semi

cenarios_didaticos = {}
for rotulo, fator_A in [("AAS idealizada", 1.0), ("Resposta seletiva: A × 0,55", 0.55)]:
    prob = VERDADE * np.array([fator_A, 1, 1, 1, 1, 1])
    prob /= prob.sum()
    contagens = rng_cobertura.multinomial(ENTREVISTAS_POR_PESQUISA, prob, size=NUM_REPETICOES)
    baixo, alto = intervalo_wilson(contagens, ENTREVISTAS_POR_PESQUISA)
    cobertura = (baixo <= VERDADE) & (VERDADE <= alto)
    cenarios_didaticos[rotulo] = {"estimativas":contagens/ENTREVISTAS_POR_PESQUISA,
                                  "cobertura":cobertura}

fig, ax = plt.subplots(figsize=(10, 5))
R = np.arange(1, NUM_REPETICOES+1)
for nome, dados in cenarios_didaticos.items():
    frequencia = 100 * np.cumsum(dados["cobertura"][:, 0]) / R
    ax.plot(R[19:], frequencia[19:], label=nome)
ax.axhline(95, linestyle="--", color="black", label="referência: 95%")
ax.set(xlabel="Número de pesquisas repetidas", ylabel="Intervalos que contêm a verdade (%)",
       title="Cobertura acumulada para o candidato fictício A")
ax.set_ylim(-2, 102)
ax.legend()
ax.grid(alpha=.2)
plt.show()
print("Coberturas finais do candidato A:")
for nome, dados in cenarios_didaticos.items():
    print(f"{nome}: {100*dados['cobertura'][:,0].mean():.2f}%")

#### Cinco procedimentos sintéticos, candidato por candidato

O CSV v1.5 contém **3.000 pesquisas por método e cenário**, todas com 2.000 entrevistas e calculadas sobre a população fixa de 1 milhão de eleitores. Se o CSV ainda não está no diretório `dados/` do repositório, coloque-o no Colab pelo painel de arquivos ou faça upload manual. A célula anterior funciona independentemente desse arquivo.

Para examinar a convergência, altere `CANDIDATO_EXAMINADO`, `FATOR_RESPOSTA_A` e `REPETICOES_VISIVEIS` e execute novamente a célula abaixo. A linha horizontal de 95% é **alvo teórico para os controles probabilísticos**, não uma meta garantida para todas as curvas.

In [ ]:
from pathlib import Path
from IPython.display import display

ARQUIVO_COBERTURA = next((p for p in [
    Path('dados/rodadas_cobertura_por_candidato_v1_5.csv.gz'),
    Path('rodadas_cobertura_por_candidato_v1_5.csv.gz'),
    Path('/content/rodadas_cobertura_por_candidato_v1_5.csv.gz'),
] if p.is_file()), None)

# Configuração do gráfico: altere e execute novamente.
CANDIDATO_EXAMINADO = 'A'         # A, B, C, OUTRA, INDECISO, BRANCO_NULO
FATOR_RESPOSTA_A = 1.00         # 1.00 (neutro) ou 0.55 (participação seletiva)
REPETICOES_VISIVEIS = 3000      # de 30 a 3000

rotulos = {
    'aleatoria_simples':'Aleatória simples (IC probabilístico)',
    'municipal_probabilistico':'Municípios PPS (IC por conglomerados)',
    'domiciliar_seletivo':'Domiciliar (pseudo-IC)',
    'fluxo_cotas_aproximado':'Fluxo com cotas, aproximação (pseudo-IC)',
    'digital_posestratificado':'Digital pós-estratificado (pseudo-IC)',
}
if ARQUIVO_COBERTURA is None:
    print('Arquivo de rodadas v1.5 não encontrado. Para os cinco métodos,')
    print('envie rodadas_cobertura_por_candidato_v1_5.csv.gz ao Colab.')
    print('O primeiro gráfico, de simulação autossuficiente, já está disponível acima.')
else:
    rodadas_95 = pd.read_csv(ARQUIVO_COBERTURA)
    assert 30 <= REPETICOES_VISIVEIS <= 3000
    d95 = rodadas_95.loc[
        (rodadas_95['candidato'] == CANDIDATO_EXAMINADO) &
        np.isclose(rodadas_95['cenario_resposta_A'], FATOR_RESPOSTA_A) &
        (rodadas_95['rep'] <= REPETICOES_VISIVEIS)
    ].copy()
    assert len(d95) == 5 * REPETICOES_VISIVEIS
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for metodo, grupo in d95.groupby('metodo'):
        grupo = grupo.sort_values('rep')
        ax.plot(grupo['rep'].to_numpy()[19:],
                100*grupo['cobertura_acumulada'].to_numpy()[19:],
                linestyle='-' if metodo in ('aleatoria_simples','municipal_probabilistico') else '--',
                label=rotulos[metodo])
    ax.axhline(95, color='black', linestyle=':', linewidth=2,
               label='95% — referência nominal')
    ax.set(xlabel='Número de pesquisas repetidas',
           ylabel='Cobertura acumulada (%)',
           title=f'Candidato {CANDIDATO_EXAMINADO} — fator de resposta A: {FATOR_RESPOSTA_A:.2f}')
    ax.set_ylim(-3,103)
    ax.grid(alpha=.2)
    ax.legend(fontsize=8, ncol=2)
    plt.show()
    print('Cobertura até a repetição', REPETICOES_VISIVEIS)
    display(d95.groupby('metodo',as_index=False).agg(
        cobertura_empirica=('cobre', lambda s:100*s.mean()),
        estimativa_media=('estimativa',lambda s:100*s.mean()),
        valor_verdadeiro=('verdade',lambda s:100*s.iloc[0])
    ).round(2))

**Perguntas para os estudantes:** (1) Por que a curva do controle aleatório oscila muito nas primeiras pesquisas? (2) A partir de quantas repetições ela se estabiliza aproximadamente? (3) Por que uma curva pode se estabilizar **longe de 95%**, mesmo com milhares de pesquisas? (4) O que muda quando selecionamos o candidato B ou C? (5) Por que a pós-estratificação demográfica não corrige necessariamente a não resposta dependente do voto?

**Precisão:** em 3.000 repetições, se a cobertura verdadeira é 95%, o desvio-padrão da frequência simulada é de aproximadamente **0,40 ponto percentual**. Por isso 94,8%, 95,2% ou 95,5% não são anomalias. Não é a **estimativa de votos** que tende a 95%: é a proporção de **intervalos contendo a verdade** que tende à cobertura real do método.

## 4. *Bootstrap*: reamostragem de respostas simuladas

Repetir uma pesquisa real demandaria selecionar e entrevistar novas pessoas. O *bootstrap* faz algo diferente: **reamostra com reposição os dados que já temos** e recalcula a estimativa.

Vamos sortear **uma amostra fictícia** de 2.000 respostas binárias e, a partir dela, obter 2.000 reamostras. A escolha da amostra e a regra de reamostragem serão inteiramente reproduzíveis.

O resultado não valida as pesquisas reais e não oferece acesso a seus microdados. Serve para compreender por que uma estatística obtida de uma amostra tem variabilidade.

In [ ]:
rng_boot_eleitoral = np.random.default_rng(20261016)
respostas_ficticias = rng_boot_eleitoral.binomial(1, p_populacao, size=2000)
B_eleitoral = 2000
estimativas_boot_ficticias = np.array([
    rng_boot_eleitoral.choice(respostas_ficticias,
                             size=len(respostas_ficticias), replace=True).mean()
    for _ in range(B_eleitoral)
])
intervalo_ficticio = np.quantile(estimativas_boot_ficticias, [0.025, 0.975])
print(f"Amostra fictícia: {respostas_ficticias.mean()*100:.1f}%")
print(f"IC bootstrap percentil didático: {intervalo_ficticio[0]*100:.1f}% a {intervalo_ficticio[1]*100:.1f}%")
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(estimativas_boot_ficticias*100, bins=30)
ax.axvline(respostas_ficticias.mean()*100, linestyle="--", label="estimativa simulada")
ax.set_xlabel("Proporção reamostrada (%)")
ax.set_title("Bootstrap: distribuição de uma estimativa fictícia")
ax.legend()
plt.show()

## 5. Outra aplicação: os 120 documentos da Unidade 5

Agora retornaremos ao corpus com **120 documentos fictícios** utilizado nos notebooks anteriores. Aqui a quantidade de interesse muda: em vez de uma proporção de intenção de voto, investigaremos a **diferença entre médias de `palavras`** dos grupos `Capital` e `Interior`.

Se `dados/documentos.csv` já existir no ambiente, será carregado. Caso contrário, o notebook reproduzirá o gerador do guia da Unidade 5, mantendo a mesma semente.

A distinção é indispensável: **um corpus sintético não é uma amostra probabilística de documentos históricos**. Intervalos e testes a seguir ilustram procedimentos matemáticos condicionais; não autorizam inferência histórica sobre uma população não definida.

In [ ]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ARQUIVO = Path("dados/documentos.csv")

def gerar_dados():
    SEED, N = 20261058, 120
    rng = np.random.default_rng(SEED)
    linhas = []
    for ano in range(1890, 1910):
        t = (ano - 1890) / 19
        p_capital = 0.12 + 0.76 * t
        n_capital = int(np.clip(rng.binomial(6, p_capital), 1, 5))
        locais_ano = np.array(["Capital"] * n_capital + ["Interior"] * (6 - n_capital))
        rng.shuffle(locais_ano)
        linhas.extend((ano, local) for local in locais_ano)

    rng.shuffle(linhas)
    anos = np.array([x[0] for x in linhas])
    locais = np.array([x[1] for x in linhas])
    capital = locais == "Capital"

    generos = rng.choice(["notícia", "editorial", "carta"], N, p=[0.45, 0.30, 0.25])
    temas_possiveis = np.array(["educação", "trabalho", "progresso", "saúde"])
    probs = {
        "notícia": [0.20, 0.25, 0.15, 0.40],
        "editorial": [0.20, 0.30, 0.35, 0.15],
        "carta": [0.40, 0.25, 0.20, 0.15],
    }
    temas = np.array([rng.choice(temas_possiveis, p=probs[g]) for g in generos])

    efeito_genero = {"notícia": 0.03, "editorial": 0.24, "carta": -0.18}
    log_palavras = (
        math.log(650)
        + np.array([efeito_genero[g] for g in generos])
        + np.where(capital, 0.64, 0.0)
        - 0.016 * (anos - 1890)
        + rng.normal(0, 0.24, N)
    )
    palavras = np.maximum(140, np.rint(np.exp(log_palavras))).astype(int)

    especiais = rng.choice(N, 3, replace=False)
    palavras[especiais] = np.rint(
        palavras[especiais] * rng.uniform(1.65, 2.05, 3)
    ).astype(int)

    densidade = np.clip(rng.normal(365, 38, N), 270, 470)
    paginas = np.maximum(1, np.rint(palavras / densidade)).astype(int)

    taxa = {"notícia": 7.0, "editorial": 4.7, "carta": 5.8}
    pessoas = rng.poisson(palavras / 1000 * np.array([taxa[g] for g in generos]))

    df = pd.DataFrame({
        "id_documento": [f"R{i:03d}" for i in range(1, N + 1)],
        "ano": anos, "genero": generos, "local": locais, "tema": temas,
        "palavras": palavras, "paginas": paginas, "pessoas": pessoas,
    })
    ARQUIVO.parent.mkdir(exist_ok=True)
    df.to_csv(ARQUIVO, index=False)
    return df

if ARQUIVO.exists():
    dados = pd.read_csv(ARQUIVO)
    print("Dados carregados de:", ARQUIVO)
else:
    dados = gerar_dados()
    print("Dados reproduzidos em:", ARQUIVO)

print("Dimensões:", dados.shape)
display(dados.head())

## 6. *Bootstrap* para a diferença entre médias

No Notebook 01, a estimativa principal foi a diferença entre as médias de `palavras` dos grupos `Capital` e `Interior`.

$$
\Delta_{\mathrm{obs}}
=
\bar{x}_{Capital}
-
\bar{x}_{Interior}.
$$

O *bootstrap* procura responder: se pudéssemos repetir um procedimento semelhante ao que produziu os dados observados, quanto essa diferença tenderia a variar?

Em cada repetição $b$, reamostramos com reposição dentro de cada grupo e calculamos:

$$
\Delta_b
=
\bar{x}_{Capital,b}
-
\bar{x}_{Interior,b}.
$$

A coleção desses valores forma uma **distribuição bootstrap da estimativa**. Ela não representa novos passados possíveis; representa a variação produzida pela regra de reamostragem sobre os dados observados.

In [ ]:
a = dados.loc[dados["local"].eq("Capital"), "palavras"].to_numpy()
b = dados.loc[dados["local"].eq("Interior"), "palavras"].to_numpy()

diferenca_observada = a.mean() - b.mean()

rng = np.random.default_rng(20260906)
B = 4000
diferencas_bootstrap = np.array([
    rng.choice(a, size=len(a), replace=True).mean()
    - rng.choice(b, size=len(b), replace=True).mean()
    for _ in range(B)
])

print(f"Diferença observada: {diferenca_observada:.2f} palavras")
pd.Series(diferencas_bootstrap).describe(percentiles=[0.025, 0.5, 0.975])

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(diferencas_bootstrap, bins=35)
plt.axvline(diferenca_observada, linestyle="--", label="diferença observada")
plt.xlabel("Diferença de médias: Capital - Interior")
plt.ylabel("Frequência")
plt.title("Distribuição bootstrap da diferença")
plt.legend()
plt.show()

A largura da distribuição indica quanto a estimativa varia sob a regra de reamostragem adotada. Uma distribuição mais estreita sugere maior precisão **dentro desse procedimento**; isso não significa, por si só, maior validade histórica do corpus.

## 7. Intervalo de confiança para a diferença de médias

Usaremos o intervalo percentil introdutório:

$$
IC_{95\%}
=
\left[
Q_{0{,}025}(\Delta_b),
Q_{0{,}975}(\Delta_b)
\right].
$$

Em repetições do procedimento, cerca de 95% dos intervalos construídos dessa maneira cobririam o parâmetro-alvo sob as condições do método.

Depois que um intervalo específico foi calculado, não dizemos que há 95% de probabilidade frequentista de o parâmetro fixo estar nele. O intervalo também **não contém 95% dos documentos**.

In [ ]:
limite_inferior, limite_superior = np.quantile(
    diferencas_bootstrap, [0.025, 0.975]
)

tabela_intervalo = pd.DataFrame([{
    "estimativa": diferenca_observada,
    "limite inferior": limite_inferior,
    "limite superior": limite_superior,
    "unidade": "palavras por documento",
}])

display(tabela_intervalo.round(2))

O intervalo deve ser lido junto com a estimativa pontual, a unidade da variável, o desenho de obtenção dos dados e as fontes de incerteza que o método não representa.

Um intervalo estreito em uma coleção enviesada continua sendo um intervalo estreito sobre uma coleção enviesada.

## 8. Hipótese nula e teste de permutação

O *bootstrap* foi usado para estudar a variabilidade da estimativa. O teste de permutação responde a outra pergunta: quão incompatível é o resultado observado com uma **hipótese nula específica**?

A hipótese nula didática será:

> sob o modelo nulo, os rótulos `Capital` e `Interior` são intercambiáveis em relação à variável `palavras`.

Mantemos valores e tamanhos dos grupos, embaralhamos os rótulos e recalculamos a diferença.

Para $M$ permutações:

$$
p
=
\frac{
1+\sum_{m=1}^{M}
\mathbf{1}\left(|T_m|\geq|T_{obs}|\right)
}{
M+1
}.
$$

Usamos valores absolutos porque o teste é bilateral: resultados extremos nas duas direções contam contra o modelo nulo.

In [ ]:
rng = np.random.default_rng(20260906)
combinados = np.concatenate([a, b])
M = 5000
diferencas_nulas = np.empty(M)

for m in range(M):
    embaralhados = rng.permutation(combinados)
    diferencas_nulas[m] = (
        embaralhados[:len(a)].mean()
        - embaralhados[len(a):].mean()
    )

valor_p = (
    1 + np.sum(np.abs(diferencas_nulas) >= abs(diferenca_observada))
) / (M + 1)

display(pd.Series({
    "diferença observada": diferenca_observada,
    "valor de p bilateral": valor_p,
}).round(4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(diferencas_nulas, bins=35)
plt.axvline(diferenca_observada, linestyle="--", label="diferença observada")
plt.axvline(-diferenca_observada, linestyle=":", label="valor simétrico")
plt.xlabel("Diferença de médias sob permutação")
plt.ylabel("Frequência")
plt.title("Distribuição nula por permutação")
plt.legend()
plt.show()

## 9. Como interpretar o valor de *p*

O valor de *p* responde a uma pergunta condicional:

> **Se a hipótese nula e o procedimento especificado fossem adequados, quão extremos seriam resultados como o observado?**

Ele **não** informa:

- a probabilidade de a hipótese nula ser verdadeira;
- a probabilidade de a hipótese alternativa ser verdadeira;
- o tamanho da diferença;
- a importância substantiva da diferença;
- a probabilidade de replicação;
- a ausência de viés;
- a qualidade do corpus;
- a qualidade da operacionalização.

Por isso, um valor pequeno de *p* não substitui a estimativa, o tamanho de efeito, o intervalo ou a interpretação substantiva. Também não devemos transformar automaticamente `0,05` em uma fronteira entre “verdadeiro” e “falso”.

## 10. Significância estatística e relevância substantiva

Uma diferença muito pequena pode apresentar valor de *p* baixo quando há muitos dados e pouca variabilidade. Uma diferença substantivamente importante pode permanecer imprecisa quando o número de observações é pequeno ou os dados são muito heterogêneos.

Portanto:

$$
\text{valor de }p
\neq
\text{tamanho de efeito}
\neq
\text{relevância substantiva}.
$$

O valor de *p* também não corrige o processo de seleção dos dados. Se o corpus é de conveniência, a permutação continua operando sobre essa mesma coleção.

## 11. Reunindo as peças

A análise quantitativa possui componentes diferentes:

1. **estimativa** — qual diferença observamos?;
2. **tamanho de efeito** — qual a magnitude da diferença em relação à variabilidade?;
3. **incerteza** — quanto a estimativa varia sob um procedimento de reamostragem?;
4. **teste de hipótese** — quão incompatível é o resultado com um modelo nulo?;
5. **relevância substantiva** — o que a diferença significa para a pergunta de pesquisa?

Nenhuma dessas dimensões substitui as demais.

### Retorno aos três institutos: o que **não** podemos calcular com os dados publicados?

A tabela pública contém percentuais e informações do levantamento, mas **não as respostas individuais, os pesos nem todos os elementos do desenho amostral**. Por isso:

- não podemos executar um *bootstrap* fiel para Quaest, Datafolha ou AtlasIntel a partir daqueles percentuais agregados;
- não podemos tratar as margens divulgadas como se fossem obtidas pela nossa fórmula didática de amostragem aleatória simples;
- não podemos atribuir a diferença observada entre institutos a uma única causa sem examinar metodologia e datas;
- não podemos usar diretamente o teste de permutação entre `Capital` e `Interior` como teste da diferença de intenções de voto de dois candidatos na **mesma amostra**.

Para uma comparação mais rigorosa, precisaríamos das fichas de registro, questionários, critérios de seleção e ponderação; para recalcular a incerteza segundo o desenho, seriam necessários os dados e os procedimentos adequados.

**Perguntas de encerramento do estudo de caso:** quais diferenças podem decorrer de amostragem? Quais decorrem potencialmente da modalidade de entrevista, do período de campo, de decisões de ponderação e da forma da pergunta? O que permanece indeterminado?

## U05-A03 — Atividade integrada: ficha de leitura crítica

**Modalidade:** duplas. **Tempo sugerido:** 30–40 minutos. A atividade separa rigorosamente **pesquisas publicadas** e **experimentos simulados**.

**Parte A — pesquisas reais**

1. Escolha um dos três institutos e informe: período de coleta, tamanho da amostra, modalidade de entrevista, margem anunciada, confiança e registro no TSE.
2. Explique em suas palavras o significado de estimativa, margem e confiança, sem confundir as três informações.
3. Compare essa pesquisa com uma das outras duas. Indique pelo menos duas fontes possíveis de discrepância que **não** são resolvidas apenas com uma amostra maior.
4. Escolha uma informação metodológica que ainda precisaria ser consultada na ficha do TSE e explique por quê.

**Parte B — simulações**

5. Compare os resultados simulados para 400 e 5.000 observações: como mudou a margem aproximada?
6. Na simulação com participação desigual, a estimativa média aproximou-se do valor verdadeiro ao aumentar $n$? Explique.
7. Diferencie repetição de uma pesquisa, *bootstrap* de respostas disponíveis e teste de permutação.

**Parte C — documentos fictícios**

8. Informe a diferença observada entre `Capital` e `Interior`, seu intervalo *bootstrap* e o valor de *p* do teste didático.
9. Explique por que esses resultados não autorizam generalização histórica sobre documentos de uma população não definida.

**Minha ficha:** Escreva aqui.

**O que os cálculos permitem afirmar:** Escreva aqui.

**O que os cálculos não permitem afirmar:** Escreva aqui.

### Fontes e limites da evidência

As estimativas dos três institutos são resultados **publicados**, transcritos com indicação de fonte; não se trata de microdados abertos de respondentes. Os registros da pesquisa podem ser pesquisados no [Portal de Dados Abertos do TSE](https://dadosabertos.tse.jus.br/dataset/pesquisas-eleitorais-2026). A plataforma disponibiliza registros e documentos, mas esses recursos não equivalem necessariamente a dados individuais ponderados.

A checagem integral das fichas primárias dos três registros (inclusive da discrepância de data da Quaest) é uma etapa de auditoria documental separada. Em especial, **não devemos atribuir as diferenças entre levantamentos somente aos métodos de coleta**.

O eixo quantitativo encerra-se aqui. No Notebook 03, o problema passa a ser como representar os documentos e comparar propriedades textuais.